In [1]:
## This notebook generates the tomographic bin edges based on
## 1) truenz/specemulatenz
## 2) *equipopulated [/equalcomoving/equalredshiftwidth in the other notebook]
## 3) num of bins

In [2]:
import h5py
import pyarrow.parquet as pq
import numpy as np
import itertools
import pandas as pd
from roman_sompz.rail_sompz.src.rail.estimation.algos.sompz import bin_assignment_spec, tomo_bins_wide_2d

In [3]:
basedir_truenz = '../job/tomobinning_project_output/true_nz/base/'
basedir_specemulatenz = '../job/tomobinning_project_output/spec_emulate_nz/base/'

In [4]:
## Choose your settings here
basedir = basedir_truenz

## Equipopulated

In [5]:
if basedir == basedir_specemulatenz:
    spec_data = pd.read_parquet('../job/data/hlwas_cosmos_260120_UM_2026_01_22_classic_specmatch.parquet')
elif basedir == basedir_truenz:
    spec_data = pd.read_parquet('../job/data/hlwas_cosmos_260120_UM_2026_01_22.parquet')
    
with h5py.File(basedir + 'specdata_deep_assignment.hdf5', 'r') as f:
    cell_deep_spec_data = f['cells'][:]
    deep_som_size = f['som_size'][0]

with h5py.File(basedir + 'specdata_wide_assignment.hdf5', 'r') as f:
    cell_wide_spec_data = f['cells'][:]
    wide_som_size = f['som_size'][0]

with h5py.File(basedir + 'nz_info/pchat.hdf5', 'r') as f:
    pchat = f['pchat'][:]
    
with h5py.File(basedir + 'nz_info/FINAL_NZ.hdf5', 'r') as f:
    zbinsc = f['meta/xvals'][0]
    
key = 'specz'

spec_data_for_pz = pd.DataFrame({key: spec_data[key],
                                 'cell_deep': cell_deep_spec_data,
                                 'cell_wide': cell_wide_spec_data})

dz = zbinsc[1] - zbinsc[0]                
bin_edge_min = 0.-dz/2.  

### First iteration
### Initial estimate of bin edge with ~ equal num galaxy per bin

In [6]:
# Define bin edges - four edge if three bins etc.
bin_edges_ini = [bin_edge_min,1.025,1.525,3]   #CHANGE THIS
bin_edges_ini = [round(bin_edge_min + round((e - bin_edge_min) / dz) * dz, 5) for e in bin_edges_ini]
print('bin_edges: ', bin_edges_ini)
numbin = len(bin_edges_ini) - 1

bin_edges:  [np.float64(-0.025), np.float64(1.025), np.float64(1.525), np.float64(2.975)]


In [7]:
tomo_bins_wide_dict = bin_assignment_spec(spec_data_for_pz,
                                          deep_som_size,
                                          wide_som_size,
                                          bin_edges = bin_edges_ini,
                                          key_z=key,
                                          key_cells_wide='cell_wide')

frac_perbin = [np.sum(pchat[tomo_bins_wide_dict[i]]) for i in range(numbin)]
frac_perbin = frac_perbin/np.sum(frac_perbin)

In [8]:
print('Want to achieve per bin: ', 1.0 / numbin)
print('What we have: ',frac_perbin)

Want to achieve per bin:  0.3333333333333333
What we have:  [0.33758605 0.32884903 0.33356492]


### Second iteration
### Expore around the initial estimate and output final bin edge

In [9]:
#Generate all posible bin edges

dz = zbinsc[1] - zbinsc[0]                
bin_edge_min = 0.-dz/2.       
specz_max = np.max(spec_data_for_pz[key])
n_steps = np.ceil((specz_max - bin_edge_min) / dz)
bin_edge_max = round(bin_edge_min + n_steps * dz, 5)


grid_points = []
current_edge = round(bin_edge_min + dz, 5)

while current_edge < bin_edge_max:
    grid_points.append(current_edge)
    current_edge = round(current_edge + dz, 5)

valid_combinations = itertools.combinations(grid_points, numbin)

bin_edges_all = []
for combo in valid_combinations:
    bin_edges_all.append([bin_edge_min] + list(combo))


In [10]:
explore_dz_scale = 0.10          #CHANGE THIS
padding = 0.001
explore_dz_scale += padding

filtered_edges = []

for bin_edges in bin_edges_all:
    is_valid = True
    
    for i in range(1, len(bin_edges)):
        lower_bound = round(bin_edges_ini[i] - explore_dz_scale, 5)
        upper_bound = round(bin_edges_ini[i] + explore_dz_scale, 5)
        
        if not (lower_bound <= bin_edges[i] <= upper_bound):
            is_valid = False
            break
            
    if is_valid:
        filtered_edges.append(bin_edges)

bin_edges_all = filtered_edges
print('Exploring this number of bin edges, reduce scale if this is >1000 ')
print(len(bin_edges_all))

Exploring this number of bin edges, reduce scale if this is >1000 
100


In [11]:
target_frac = 1.0 / numbin

best_bin_edges = None
best_frac_perbin = None
min_error = np.inf
best_tomo_bins_wide_dict = None

i = 0
for bin_edges in bin_edges_all:
    i +=1
    if i%1000 == 0:
        print('hi')
    tomo_bins_wide_dict = bin_assignment_spec(spec_data_for_pz,
                                              deep_som_size,
                                              wide_som_size,
                                              bin_edges = bin_edges,
                                              key_z=key,
                                              key_cells_wide='cell_wide')
    
    frac_perbin = [np.sum(pchat[tomo_bins_wide_dict[i]]) for i in range(numbin)]
    frac_perbin = frac_perbin/np.sum(frac_perbin)
    error = np.sum((frac_perbin - target_frac) ** 2)

    if error < min_error:
        min_error = error
        best_bin_edges = bin_edges
        best_frac_perbin = frac_perbin
        best_tomo_bins_wide_dict = tomo_bins_wide_dict

In [12]:
print("Optimal Bin Edges:", best_bin_edges)
print("Fractions for these edges:", best_frac_perbin)
print("Error score:", min_error)

Optimal Bin Edges: [np.float64(-0.025), np.float64(1.025), np.float64(1.525), np.float64(2.875)]
Fractions for these edges: [0.33758605 0.32884903 0.33356492]
Error score: 3.824825863548901e-05


In [13]:
#Paste the below in config.yaml after line 268 bin_edges
for edge in best_bin_edges:
    print(f"  - {edge}")

  - -0.025
  - 1.025
  - 1.525
  - 2.875


In [14]:
best_tomo_bins_wide_dict

{0: array([  20,   21,   22,   23,   24,   25,   26,   27,   28,   29,   30,
          31,   53,   54,   55,   56,   57,   58,   59,   60,   61,   62,
          63,   85,   86,   87,   88,   89,   90,   91,   92,   93,   94,
          95,  118,  119,  120,  121,  122,  123,  124,  125,  126,  127,
         150,  151,  152,  153,  154,  155,  156,  157,  158,  159,  182,
         183,  184,  185,  186,  187,  188,  189,  190,  191,  215,  216,
         217,  218,  219,  220,  221,  222,  223,  247,  248,  249,  250,
         251,  252,  253,  254,  255,  279,  280,  281,  282,  283,  284,
         285,  286,  287,  312,  313,  314,  315,  316,  317,  318,  319,
         344,  345,  346,  347,  348,  349,  350,  351,  357,  377,  378,
         379,  380,  381,  382,  383,  411,  412,  413,  414,  415,  437,
         443,  444,  445,  446,  447,  448,  449,  450,  457,  469,  475,
         476,  477,  478,  479,  480,  481,  489,  490,  507,  508,  509,
         510,  511,  521,  538,  53